# Document Intelligence Refinery — Run in Colab (Interim)

This notebook clones your repo (or uses the current folder), installs deps, and runs the interim pipeline: **12 DocumentProfiles** + **extraction_ledger.jsonl**.

**Before running:** Upload your `data.zip` (PDFs inside a `data/` folder) when prompted, or use a repo that already has `data/`.

## 1. Clone repo from GitHub (or skip if you opened from GitHub)

Paste your repo URL below and run. If you opened this notebook from GitHub, you can skip to section 2.

In [ ]:
# Paste your GitHub repo URL (e.g. https://github.com/username/document_intelligence)
REPO_URL = "https://github.com/YOUR_USERNAME/document_intelligence"  # <-- EDIT THIS

import os
if not os.path.exists("/content/document_intelligence/src"):
    !git clone {REPO_URL} /content/document_intelligence
    %cd /content/document_intelligence
else:
    %cd /content/document_intelligence
    !pwd

## 2. Install dependencies

In [ ]:
%cd /content/document_intelligence
!pip install -e . -q
!pip install pymupdf pyyaml -q
# Docling is heavy; install only if you need Strategy B / Phase 0 Docling
# !pip install docling -q
print("Done.")

## 3. Upload your data (PDFs)

If your repo does **not** contain the PDFs (they're in .gitignore), upload a zip of your `data/` folder here. The zip should contain the PDFs at the root (e.g. `data.zip` with `CBE ANNUAL REPORT 2023-24.pdf`, `Audit Report - 2023.pdf`, etc. inside).

In [ ]:
from google.colab import files
import os

DATA_DIR = "/content/document_intelligence/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Check if we already have PDFs (e.g. from clone)
pdfs = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".pdf")]
if len(pdfs) >= 4:
    print(f"Found {len(pdfs)} PDFs in data/. Skipping upload.")
else:
    print("Upload your data.zip (contains the PDFs).")
    uploaded = files.upload()  # Opens file picker
    for name, _ in uploaded.items():
        if name.endswith(".zip"):
            import subprocess
            subprocess.run(["unzip", "-o", "-q", name, "-d", "/content/document_intelligence"], check=True)
            os.remove(name)
            print("Unzipped.")
            break

## 4. Phase 0: Character density analysis (pdfplumber)

In [ ]:
%cd /content/document_intelligence
!python scripts/phase0_pdfplumber_analysis.py

## 5. Generate 12 profiles + extraction ledger (interim deliverables)

In [ ]:
%cd /content/document_intelligence
!python scripts/run_interim_artifacts.py

## 6. Run tests

In [ ]:
%cd /content/document_intelligence
!pip install -e ".[dev]" -q
!pytest tests/ -v --tb=short

## 7. Download artifacts

Zip `.refinery/` and download it to your machine.

In [ ]:
%cd /content/document_intelligence
!zip -r refinery_artifacts.zip .refinery
from google.colab import files
files.download("refinery_artifacts.zip")